# 05 · Models and training
Eleven agents learn from simulation data; their labels come from the simulation itself (no human labelling).
All use **gradient-boosted decision trees** (GBDT): HistGradientBoosting by default, LightGBM for incident
detection and XGBoost for ambulance arrival times, chosen with the model benchmark
(`config/agents.yaml > model_algorithms`).

* **Classifiers** answer yes/no (is this lane blocked?) → measured with **F1** (precision and recall).
* **Regressors** predict a number (seconds until the ambulance crosses) → measured with **MAE**.

Every model is compared with the hand-written **rule** it replaces, which remains the agent's fallback.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

## 1. Every trained model against its rule

In [ ]:
import json
m = json.load(open(config.path("models", "metrics.json")))
rows = []
for name, v in m.items():
    if name.startswith("_") or not isinstance(v, dict): continue
    if "f1" in v:   rows.append({"model": name, "algorithm": v.get("algorithm", "hist_gb"), "metric": "F1 (higher is better)", "model score": v["f1"], "rule": v.get("rule_f1")})
    elif "mae" in v: rows.append({"model": name, "algorithm": v.get("algorithm", "hist_gb"), "metric": "MAE (lower is better)", "model score": v["mae"], "rule": v.get("rule_mae")})
print("Trained with:", m.get("_versions"))
pd.DataFrame(rows)

## 2. Training one classifier step by step: incident detection
Each row is a lane snapshot; the label says whether the lane was really blocked by the hidden incident.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, ConfusionMatrixDisplay
from traffix.training.algorithms import make_model
df = pd.read_csv(config.path("data", "training", "incident.csv")).dropna()
X, y = df.drop(columns="label"), df["label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(f"{len(df)} snapshots, {y.mean():.1%} are real incidents")
rule = (Xte.iloc[:, 0] >= 1).astype(int)                       # the fallback rule
model, algo = make_model("incident", "clf"); model.fit(Xtr.values, ytr.values)
pred = model.predict(Xte.values)
pd.DataFrame([
    {"method": "rule (fallback)", "precision": precision_score(yte, rule), "recall": recall_score(yte, rule), "F1": f1_score(yte, rule)},
    {"method": f"{algo} (trained)", "precision": precision_score(yte, pred), "recall": recall_score(yte, pred), "F1": f1_score(yte, pred)},
]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_predictions(yte, pred, ax=axes[0], display_labels=["normal", "blocked"], colorbar=False)
axes[0].set_title(f"{algo}: confusion matrix (test set)")
from sklearn.inspection import permutation_importance
sample = Xte.sample(min(3000, len(Xte)), random_state=0)
imp = permutation_importance(model, sample.values, yte.loc[sample.index].values, scoring="f1", n_repeats=3, random_state=0)
pd.Series(imp.importances_mean, index=X.columns).sort_values().plot.barh(ax=axes[1], color="#2bc4b5")
axes[1].set_title("Which inputs matter (permutation importance)"); plt.tight_layout(); plt.show()

## 3. A regressor: when will the ambulance cross the junction?
The rule divides distance by speed, which is badly wrong when the ambulance is stuck behind a queue.

In [ ]:
import joblib
d = pd.read_csv(config.path("data", "training", "ambulance_eta.csv")).dropna()
X, y = d.iloc[:, :-1].values, d.iloc[:, -1].values
_, Xte, _, yte = train_test_split(X, y, test_size=0.25, random_state=0)
eta_model = joblib.load(config.path("models", "ambulance_eta.joblib"))
p_model, p_rule = eta_model.predict(Xte), Xte[:, 0] / np.maximum(Xte[:, 1], 1.0)
print(f"{type(eta_model).__name__}: mean error {np.mean(np.abs(p_model - yte)):.1f} s  ·  rule: {np.mean(np.abs(p_rule - yte)):.1f} s")
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(yte, p_rule, s=10, alpha=.5, label="rule", color="#9e9e9e")
ax.scatter(yte, p_model, s=10, alpha=.7, label="model", color="#1f6f6b")
lim = np.percentile(yte, 98); ax.plot([0, lim], [0, lim], "k--", lw=1)
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_xlabel("actual seconds"); ax.set_ylabel("predicted seconds"); ax.legend()
ax.set_title("Ambulance arrival time (test set)"); plt.show()

## 4. The demand forecast: trained on 2025, tested on Ramadan 2026

In [ ]:
import random
from traffix.training.train_models import forecast_dataset
Xf, yf, _, ram = forecast_dataset(2026, random.Random(0))
fc = joblib.load(config.path("models", "demand_forecast.joblib"))
level_model = fc["level"] if isinstance(fc, dict) else fc
pf = level_model.predict(Xf)
print(f"Forecast error, all of 2026: {np.mean(np.abs(pf - yf)):.3f}  ·  during Ramadan 2026: {np.mean(np.abs(pf[ram] - yf[ram])):.3f}")
idx = np.where(ram)[0][:48 * 3]
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(yf[idx], label="actual (simulated calendar)", lw=2); ax.plot(pf[idx], label="forecast", lw=1.5, ls="--")
ax.set_title("First three days of Ramadan 2026, every 30 minutes"); ax.set_xlabel("half-hours"); ax.legend(); ax.grid(alpha=.3); plt.show()

## 5. Model benchmark: is GBDT the right family?
Seven algorithms per task, same data, cross-validated (`python scripts/benchmark.py models`).
Dark green = the algorithm the agent uses.

In [ ]:
s = nt.load_benchmark("models_summary.csv")
if s is not None:
    display(s)
    p = config.path("results", "benchmark", "models.png")
    if os.path.exists(p): display(Image(p))
else:
    print("Not run yet: python scripts/benchmark.py models")